In [2]:
# CELL 1 — Install
!pip install torch torchvision --quiet
!pip install torch-geometric --quiet
!pip install stable-baselines3[extra] gymnasium numpy scipy plotly --quiet

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.7/63.7 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 79.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.0/188.0 kB 17.1 MB/s eta 0:00:00


In [1]:
#  CELL 2 — Imports & Global Config
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data
from torch_geometric.nn import GCNConv
import heapq
from scipy.ndimage import gaussian_filter
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import gymnasium as gym
from gymnasium import spaces
from stable_baselines3 import PPO
from stable_baselines3.common.env_checker import check_env
import warnings
warnings.filterwarnings('ignore')

np.random.seed(42)
torch.manual_seed(42)

GRID  = 64          # grid size (use 32 for fast testing)
# START = (3, 3)
# GOAL  = (GRID-4, GRID-4)
START = (2, 2)
GOAL  = (GRID-3, GRID//2)

# Cost weights
ALPHA, BETA, GAMMA, DELTA = 0.4, 0.3, 0.2, 0.1

print("✅ Imports done")

ModuleNotFoundError: No module named 'torch_geometric'

In [4]:
# CELL 3 — Terrain & Environment Simulation
def generate_elevation(size=GRID):
    raw = np.random.rand(size, size)
    # Add gaussian mountain peaks for realism
    peaks = [
        (size//4,   size//4,   0.9, size//8),
        (3*size//4, 3*size//4, 1.0, size//7),
        (size//2,   3*size//4, 0.7, size//9),
        (size//4,   3*size//4, 0.6, size//9),
        (3*size//4, size//4,   0.8, size//8),
    ]
    for pi, pj, h, r in peaks:
        for i in range(size):
            for j in range(size):
                d = np.sqrt((i-pi)**2 + (j-pj)**2)
                raw[i, j] += h * np.exp(-d**2 / (2*r**2))
    sm = gaussian_filter(raw, sigma=4)
    return (sm - sm.min()) / (sm.max() - sm.min())

def compute_slope(elev):
    gy, gx = np.gradient(elev)
    s = np.sqrt(gx**2 + gy**2)
    return s / s.max()

def compute_river_mask(elev, thr=0.22):
    return (elev < thr).astype(float)

def generate_wind_field(size=GRID):
    nx = gaussian_filter(np.random.randn(size, size), sigma=5) * 0.3 + 0.6
    ny = gaussian_filter(np.random.randn(size, size), sigma=5) * 0.3 + 0.4
    return nx, ny

def generate_radar_map(elev, n=4, size=GRID):
    # Place radars on high ground away from start/goal
    positions = []
    while len(positions) < n:
        ri, rj = np.random.randint(8, size-8), np.random.randint(8, size-8)
        if elev[ri, rj] > 0.5:  # prefer high ground
            positions.append((ri, rj))
    det = np.zeros((size, size))
    for ri, rj in positions:
        for i in range(size):
            for j in range(size):
                dist = np.sqrt((i-ri)**2 + (j-rj)**2) + 1e-6
                los  = max(0, elev[ri, rj] - elev[i, j] + 0.3)
                det[i, j] += los / (dist * 0.08 + 1)
    det = det / det.max()
    return det, positions

elevation     = generate_elevation()
slope         = compute_slope(elevation)
river_mask    = compute_river_mask(elevation)
wind_x, wind_y = generate_wind_field()
detection_map, radar_positions = generate_radar_map(elevation)

print(f"✅ Terrain ready | Grid={GRID}x{GRID} | Radars={len(radar_positions)}")

✅ Terrain ready | Grid=64x64 | Radars=4


In [5]:
#  CELL 4 — Multi-Objective Cost Map
def wind_penalty(wx, wy, move_dir=(0, 1)):
    dy, dx = move_dir
    dot = wx * dx + wy * dy
    return np.clip(-dot, 0, 1)

def build_cost_map(elev, slope, det, wx, wy, river, move_dir=(0, 1)):
    E = slope
    D = det
    W = wind_penalty(wx, wy, move_dir)
    R = river
    C = ALPHA*E + BETA*D + GAMMA*W + DELTA*R
    return C / C.max()

cost_map = build_cost_map(elevation, slope, detection_map, wind_x, wind_y, river_mask)
print(f"✅ Cost map | min={cost_map.min():.4f}  max={cost_map.max():.4f}")

✅ Cost map | min=0.0226  max=1.0000


In [6]:
# CELL 5 — A* Planner
def heuristic(a, b):
    return np.sqrt((a[0]-b[0])**2 + (a[1]-b[1])**2)

def astar(cmap, start, goal, size=GRID):
    dirs = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
    open_set = [(0.0, start)]
    came_from = {}
    g = {start: 0.0}

    while open_set:
        _, cur = heapq.heappop(open_set)
        if cur == goal:
            path = []
            while cur in came_from:
                path.append(cur); cur = came_from[cur]
            path.append(start)
            return path[::-1]
        for d in dirs:
            nb = (cur[0]+d[0], cur[1]+d[1])
            if not (0 <= nb[0] < size and 0 <= nb[1] < size):
                continue
            tg = g[cur] + cmap[nb] * np.sqrt(d[0]**2+d[1]**2)
            if tg < g.get(nb, 1e18):
                came_from[nb] = cur
                g[nb] = tg
                heapq.heappush(open_set, (tg + heuristic(nb, goal), nb))
    return []

def path_metrics(path, cmap, det):
    energy   = sum(cmap[p] for p in path)
    exposure = sum(det[p]  for p in path)
    return energy, exposure, len(path)

astar_path = astar(cost_map, START, GOAL)
e_as, d_as, l_as = path_metrics(astar_path, cost_map, detection_map)
print(f"✅ A*  | Steps={l_as}  Energy={e_as:.2f}  Exposure={d_as:.2f}")

✅ A*  | Steps=60  Energy=38.36  Exposure=39.55


In [7]:
#  CELL 6 — Graph Neural Network
def build_graph(elev, slope, wx, wy, det, river, cmap, size=GRID):
    idx = lambda i, j: i * size + j
    feats = np.stack([
        elev.flatten(), slope.flatten(),
        wx.flatten(),   wy.flatten(),
        det.flatten(),  river.flatten()
    ], axis=1).astype(np.float32)

    src, dst = [], []
    for i in range(size):
        for j in range(size):
            for di, dj in [(-1,0),(1,0),(0,-1),(0,1),(-1,-1),(-1,1),(1,-1),(1,1)]:
                ni, nj = i+di, j+dj
                if 0 <= ni < size and 0 <= nj < size:
                    src.append(idx(i,j)); dst.append(idx(ni,nj))

    return Data(
        x          = torch.tensor(feats),
        edge_index = torch.tensor([src, dst], dtype=torch.long),
        y          = torch.tensor(cmap.flatten(), dtype=torch.float)
    )

class TerrainGNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.c1  = GCNConv(6, 64)
        self.c2  = GCNConv(64, 64)
        self.c3  = GCNConv(64, 32)
        self.bn1 = nn.BatchNorm1d(64)
        self.bn2 = nn.BatchNorm1d(64)
        self.head= nn.Linear(32, 1)

    def forward(self, x, ei):
        x = F.relu(self.bn1(self.c1(x, ei)))
        x = F.relu(self.bn2(self.c2(x, ei)))
        x = F.relu(self.c3(x, ei))
        return torch.sigmoid(self.head(x)).squeeze(-1)

gdata  = build_graph(elevation, slope, wind_x, wind_y, detection_map, river_mask, cost_map)
gnn    = TerrainGNN()
opt    = torch.optim.Adam(gnn.parameters(), lr=1e-3)
losses = []

gnn.train()
for ep in range(600):
    opt.zero_grad()
    pred = gnn(gdata.x, gdata.edge_index)
    loss = F.mse_loss(pred, gdata.y)
    loss.backward(); opt.step()
    losses.append(loss.item())
    if ep % 50 == 0:
        print(f"  Epoch {ep:3d}  Loss={loss.item():.6f}")

gnn.eval()
with torch.no_grad():
    gnn_risk = gnn(gdata.x, gdata.edge_index).numpy().reshape(GRID, GRID)

print(f"✅ GNN trained | Final loss={losses[-1]:.6f}")

  Epoch   0  Loss=0.049325
  Epoch  50  Loss=0.000642
  Epoch 100  Loss=0.000417
  Epoch 150  Loss=0.000350
  Epoch 200  Loss=0.000310
  Epoch 250  Loss=0.000281
  Epoch 300  Loss=0.000256
  Epoch 350  Loss=0.000234
  Epoch 400  Loss=0.000215
  Epoch 450  Loss=0.000198
  Epoch 500  Loss=0.000183
  Epoch 550  Loss=0.000169
✅ GNN trained | Final loss=0.000156


In [8]:
 # CELL 7 — GNN-Guided A*
# gnn_cost = (0.5 * cost_map + 0.5 * gnn_risk)
# gnn_cost = gnn_cost / gnn_cost.max()

# gnn_path = astar(gnn_cost, START, GOAL)
gnn_cost = gnn_risk / gnn_risk.max()
gnn_path = astar(gnn_cost, START, GOAL)
e_gn, d_gn, l_gn = path_metrics(gnn_path, cost_map, detection_map)
print(f"✅ GNN A* | Steps={l_gn}  Energy={e_gn:.2f}  Exposure={d_gn:.2f}")

✅ GNN A* | Steps=60  Energy=38.54  Exposure=39.51


In [16]:
# CELL 8 — RL Environment (with A* path guidance) [ new ]
class DroneNavEnv(gym.Env):
    metadata = {'render_modes': []}

    def __init__(self, cmap, grisk, det, start, goal, grid=GRID, guide_path=None):
        super().__init__()
        self.cmap       = cmap
        self.grisk      = grisk
        self.det        = det
        self.start      = tuple(start)
        self.goal       = tuple(goal)
        self.grid       = grid
        self.max_steps  = grid * 12
        self.patch      = 5
        self._maxd      = max(heuristic(start, goal), 1e-6)
        self.dirs       = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]
        # A* guide path for imitation shaping
        self.guide_path = set(guide_path) if guide_path else set()

        obs_dim = self.patch**2 * 2 + 4
        self.observation_space = spaces.Box(0.0, 2.0, shape=(obs_dim,), dtype=np.float32)
        self.action_space      = spaces.Discrete(8)

    def _patch(self, arr, pos):
        h   = self.patch // 2
        pad = np.pad(arr, h, mode='edge')
        i, j = pos[0]+h, pos[1]+h
        return pad[i-h:i+h+1, j-h:j+h+1].flatten()

    def _obs(self):
        cp = self._patch(self.cmap,  self.pos)
        gp = self._patch(self.grisk, self.pos)
        rp = np.array(self.pos,   dtype=np.float32) / self.grid
        rg = np.array(self.goal,  dtype=np.float32) / self.grid
        return np.concatenate([cp, gp, rp, rg]).astype(np.float32)

    def _near_goal(self):
        return max(abs(self.pos[0]-self.goal[0]),
                   abs(self.pos[1]-self.goal[1])) <= 2   # wider landing zone

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self.pos      = list(self.start)
        self.steps    = 0
        self.visited  = set()
        self.path_log = [tuple(self.pos)]
        self._prevd   = heuristic(self.pos, self.goal)
        return self._obs(), {}

    def step(self, action):
        d  = self.dirs[action]
        ni = int(np.clip(self.pos[0]+d[0], 0, self.grid-1))
        nj = int(np.clip(self.pos[1]+d[1], 0, self.grid-1))
        self.pos    = [ni, nj]
        self.steps += 1
        self.path_log.append(tuple(self.pos))

        curd     = heuristic(self.pos, self.goal)
        progress = (self._prevd - curd) / self._maxd
        self._prevd = curd

        cost    = float(self.cmap[ni, nj])
        det_val = float(self.det[ni, nj])
        revisit = -0.05 if tuple(self.pos) in self.visited else 0.0
        self.visited.add(tuple(self.pos))

        # Bonus for staying near the A* guide path
        guide_bonus = 0.3 if tuple(self.pos) in self.guide_path else 0.0

        reward     = (10.0 * progress
                      - 0.2  * cost
                      - 0.1  * det_val
                      + revisit
                      + guide_bonus)

        terminated = self._near_goal()
        truncated  = self.steps >= self.max_steps

        if terminated:
            reward += 150.0
        elif truncated:
            reward -= (curd / self._maxd) * 15.0

        return self._obs(), reward, terminated, truncated, {}



In [17]:
# CELL 9 — Curriculum Training (small → full grid)

# Stage 1: train on A*-guided environment (full grid, A* shapes the reward)
print("📚 Stage 1 — Training with A* guide shaping (80k steps)...")
env_s1 = DroneNavEnv(cost_map, gnn_risk, detection_map,
                     START, GOAL, guide_path=astar_path)
check_env(env_s1, warn=True)

agent = PPO(
    "MlpPolicy", env_s1,
    learning_rate = 5e-4,
    n_steps       = 2048,
    batch_size    = 128,
    n_epochs      = 15,
    gamma         = 0.999,
    gae_lambda    = 0.95,
    ent_coef      = 0.02,
    clip_range    = 0.2,
    verbose       = 0,
    policy_kwargs = dict(net_arch=[256, 256, 128])
)
agent.learn(total_timesteps=80_000)
print("   ✅ Stage 1 done")

# Stage 2: fine-tune WITHOUT guide (agent must be independent now)
print("📚 Stage 2 — Fine-tuning without guide (60k steps)...")
env_s2 = DroneNavEnv(cost_map, gnn_risk, detection_map,
                     START, GOAL, guide_path=None)
agent.set_env(env_s2)
agent.learn(total_timesteps=60_000, reset_num_timesteps=False)
print("   ✅ Stage 2 done — PPO training complete!")

📚 Stage 1 — Training with A* guide shaping (80k steps)...
   ✅ Stage 1 done
📚 Stage 2 — Fine-tuning without guide (60k steps)...
   ✅ Stage 2 done — PPO training complete!


In [18]:
# CELL 10 — Guided Rollout + A* Stitch Safety Net
def guided_rollout(agent, env, max_greedy_wait=10):
    """
    Run trained agent. If stuck, inject greedy action.
    If still fails, stitch remaining gap with A*.
    """
    obs, _ = env.reset()
    done   = False
    stuck  = 0
    prevd  = heuristic(env.pos, env.goal)

    while not done:
        action, _ = agent.predict(obs, deterministic=True)
        obs, _, term, trunc, _ = env.step(int(action))
        done = term or trunc

        curd  = heuristic(env.pos, env.goal)
        stuck = 0 if curd < prevd - 0.1 else stuck + 1
        prevd = curd

        # Greedy override when stuck
        if stuck >= max_greedy_wait and not done:
            best_a, best_v = 0, 1e9
            for a, d in enumerate(env.dirs):
                ni = int(np.clip(env.pos[0]+d[0], 0, env.grid-1))
                nj = int(np.clip(env.pos[1]+d[1], 0, env.grid-1))
                v  = heuristic([ni,nj], env.goal) + env.cmap[ni,nj] * 3
                if v < best_v:
                    best_v, best_a = v, a
            obs, _, term, trunc, _ = env.step(best_a)
            done  = term or trunc
            stuck = 0

    return env.path_log, env._near_goal()


# Run rollout on a fresh env
eval_env              = DroneNavEnv(cost_map, gnn_risk, detection_map, START, GOAL)
rl_path, reached      = guided_rollout(agent, eval_env)

# Safety net — stitch with A* if still not reached
if not reached:
    last    = rl_path[-1]
    tail    = astar(gnn_cost, last, GOAL)
    rl_path = rl_path + tail[1:]
    print(f"  ⚠️  A* stitch applied from {last} → {GOAL} ({len(tail)} steps)")
else:
    print("  🎯 Goal reached by RL agent directly!")

e_rl, d_rl, l_rl = path_metrics(rl_path, cost_map, detection_map)
print(f"\n✅ RL path  | Steps={l_rl}  Energy={e_rl:.2f}  Exposure={d_rl:.2f}")
print(f"   Goal reached (near): {reached}")

  ⚠️  A* stitch applied from (0, 0) → (61, 32) (62 steps)

✅ RL path  | Steps=830  Energy=476.16  Exposure=541.01
   Goal reached (near): False


In [30]:
# #  CELL 8 — RL Environment
# class DroneNavEnv(gym.Env):
#     metadata = {'render_modes': []}

#     def __init__(self, cmap, grisk, det, start, goal, grid=GRID):
#         super().__init__()
#         self.cmap      = cmap
#         self.grisk     = grisk
#         self.det       = det
#         self.start     = tuple(start)
#         self.goal      = tuple(goal)
#         self.grid      = grid
#         self.max_steps = grid * 10
#         self.patch     = 5
#         self._maxd     = heuristic(start, goal)
#         self.dirs      = [(-1,-1),(-1,0),(-1,1),(0,-1),(0,1),(1,-1),(1,0),(1,1)]

#         obs_dim = self.patch**2 * 2 + 4
#         self.observation_space = spaces.Box(0.0, 2.0, shape=(obs_dim,), dtype=np.float32)
#         self.action_space      = spaces.Discrete(8)

#     def _patch(self, arr, pos):
#         h   = self.patch // 2
#         pad = np.pad(arr, h, mode='edge')
#         i, j = pos[0]+h, pos[1]+h
#         return pad[i-h:i+h+1, j-h:j+h+1].flatten()

#     def _obs(self):
#         cp = self._patch(self.cmap,  self.pos)
#         gp = self._patch(self.grisk, self.pos)
#         rp = np.array(self.pos,      dtype=np.float32) / self.grid
#         rg = np.array(self.goal,     dtype=np.float32) / self.grid
#         return np.concatenate([cp, gp, rp, rg]).astype(np.float32)

#     def _near_goal(self):
#         return max(abs(self.pos[0]-self.goal[0]),
#                    abs(self.pos[1]-self.goal[1])) <= 1

#     def reset(self, seed=None, options=None):
#         super().reset(seed=seed)
#         self.pos      = list(self.start)
#         self.steps    = 0
#         self.visited  = set()
#         self.path_log = [tuple(self.pos)]
#         self._prevd   = heuristic(self.pos, self.goal)
#         return self._obs(), {}

#     def step(self, action):
#         d  = self.dirs[action]
#         ni = int(np.clip(self.pos[0]+d[0], 0, self.grid-1))
#         nj = int(np.clip(self.pos[1]+d[1], 0, self.grid-1))
#         self.pos    = [ni, nj]
#         self.steps += 1
#         self.path_log.append(tuple(self.pos))

#         curd     = heuristic(self.pos, self.goal)
#         progress = (self._prevd - curd) / self._maxd
#         self._prevd = curd

#         cost    = float(self.cmap[ni, nj])
#         det     = float(self.det[ni, nj])
#         revisit = -0.03 if tuple(self.pos) in self.visited else 0.0
#         self.visited.add(tuple(self.pos))

#         reward     = 8.0*progress - 0.2*cost - 0.1*det + revisit
#         terminated = self._near_goal()
#         truncated  = self.steps >= self.max_steps

#         if terminated:
#             reward += 100.0
#         elif truncated:
#             reward -= (curd / self._maxd) * 10.0

#         return self._obs(), reward, terminated, truncated, {}

# env = DroneNavEnv(cost_map, gnn_risk, detection_map, START, GOAL)
# check_env(env, warn=True)
# print("✅ RL env validated")

✅ RL env validated


In [31]:
# #  CELL 9 — Train PPO
# agent = PPO(
#     "MlpPolicy", env,
#     learning_rate = 5e-4,
#     n_steps       = 2048,
#     batch_size    = 128,
#     n_epochs      = 15,
#     gamma         = 0.998,
#     gae_lambda    = 0.95,
#     ent_coef      = 0.02,
#     clip_range    = 0.2,
#     verbose       = 1,
#     policy_kwargs = dict(net_arch=[256, 256])
# )
# print("🚀 Training PPO (120k steps) ...")
# agent.learn(total_timesteps=120_000)
# print("✅ PPO training complete")

Using cuda device
Wrapping the env with a `Monitor` wrapper
Wrapping the env in a DummyVecEnv.
🚀 Training PPO (120k steps) ...
---------------------------------
| rollout/           |          |
|    ep_len_mean     | 640      |
|    ep_rew_mean     | -142     |
| time/              |          |
|    fps             | 537      |
|    iterations      | 1        |
|    time_elapsed    | 3        |
|    total_timesteps | 2048     |
---------------------------------
-----------------------------------------
| rollout/                |             |
|    ep_len_mean          | 567         |
|    ep_rew_mean          | -79.4       |
| time/                   |             |
|    fps                  | 501         |
|    iterations           | 2           |
|    time_elapsed         | 8           |
|    total_timesteps      | 4096        |
| train/                  |             |
|    approx_kl            | 0.014309294 |
|    clip_fraction        | 0.237       |
|    clip_range           | 0

In [32]:
# # CELL 10 — Guided Rollout (PPO + greedy fallback + A* stitch)
# def guided_rollout(agent, env):
#     obs, _ = env.reset()
#     done, stuck, prevd = False, 0, heuristic(env.pos, env.goal)

#     while not done:
#         action, _ = agent.predict(obs, deterministic=True)
#         obs, _, term, trunc, _ = env.step(int(action))
#         done = term or trunc

#         curd = heuristic(env.pos, env.goal)
#         stuck = 0 if curd < prevd else stuck + 1
#         prevd = curd

#         if stuck >= 12 and not done:
#             best_a, best_v = 0, 1e9
#             for a, d in enumerate(env.dirs):
#                 ni = int(np.clip(env.pos[0]+d[0], 0, env.grid-1))
#                 nj = int(np.clip(env.pos[1]+d[1], 0, env.grid-1))
#                 v  = heuristic([ni, nj], env.goal) + env.cmap[ni, nj] * 2
#                 if v < best_v: best_v, best_a = v, a
#             obs, _, term, trunc, _ = env.step(best_a)
#             done = term or trunc
#             stuck = 0

#     return env.path_log, env._near_goal()

# rl_path, reached = guided_rollout(agent, env)

# # Stitch remaining gap with A* if needed
# if not reached:
#     tail    = astar(gnn_cost, rl_path[-1], GOAL)
#     rl_path = rl_path + tail[1:]
#     print(f"  ⚠️  Gap stitched with A* tail ({len(tail)} steps)")

# e_rl, d_rl, l_rl = path_metrics(rl_path, cost_map, detection_map)
# print(f"✅ RL path  | Steps={l_rl}  Energy={e_rl:.2f}  Exposure={d_rl:.2f}")
# print(f"   Goal reached (near): {reached}")

  ⚠️  Gap stitched with A* tail (9 steps)
✅ RL path  | Steps=649  Energy=384.57  Exposure=362.98
   Goal reached (near): False


In [19]:
# CELL 11 — Metrics Summary
print("\n" + "="*58)
print(f"{'Method':<20} {'Energy':>8} {'Exposure':>10} {'Steps':>7}")
print("-"*58)
print(f"{'A* Baseline':<20} {e_as:>8.2f} {d_as:>10.2f} {l_as:>7}")
print(f"{'GNN-Guided A*':<20} {e_gn:>8.2f} {d_gn:>10.2f} {l_gn:>7}")
print(f"{'PPO RL Agent':<20} {e_rl:>8.2f} {d_rl:>10.2f} {l_rl:>7}")
print("="*58)


Method                 Energy   Exposure   Steps
----------------------------------------------------------
A* Baseline             38.36      39.55      60
GNN-Guided A*           38.54      39.51      60
PPO RL Agent           476.16     541.01     830


In [20]:
# CELL 12 — Plotly Visualizations
grid_ax = np.arange(GRID)
X, Y    = np.meshgrid(grid_ax, grid_ax)

PATHS = {
    "A* Baseline":   (astar_path, "#ff6b6b", 25),
    "GNN-Guided A*": (gnn_path,   "#00ffff", 50),
    "PPO RL Agent":  (rl_path,    "#ffdd00", 75),
}

def pcoords(path, elev, off=30):
    return ([p[1] for p in path],
            [p[0] for p in path],
            [elev[p[0], p[1]] * 800 + off for p in path])

# ── VIZ 1 — 3D Cinematic Flight ──────────────────────────────────────────────
fig1 = go.Figure()

fig1.add_trace(go.Surface(
    x=X, y=Y, z=elevation * 800,
    surfacecolor=elevation,
    colorscale=[
        [0.00, "#0a3d0a"], [0.20, "#2d6a2d"],
        [0.45, "#6b8e23"], [0.60, "#8B7355"],
        [0.80, "#A9A9A9"], [1.00, "#FFFFFF"]
    ],
    showscale=False,
    lighting=dict(ambient=0.55, diffuse=0.85, specular=0.2, roughness=0.6),
    lightposition=dict(x=200, y=-200, z=1500),
    opacity=1.0, name="Terrain", hoverinfo="skip"
))

# Radar towers
for ri, rj in radar_positions:
    rz = elevation[ri, rj] * 800
    fig1.add_trace(go.Scatter3d(
        x=[rj,rj], y=[ri,ri], z=[rz, rz+60],
        mode="lines", line=dict(color="red", width=4),
        showlegend=False, hoverinfo="skip"
    ))
    fig1.add_trace(go.Scatter3d(
        x=[rj], y=[ri], z=[rz+70],
        mode="markers+text",
        marker=dict(size=7, color="red", symbol="diamond"),
        text=["📡"], textposition="top center",
        showlegend=False, hoverinfo="skip"
    ))

# Flight paths + shadows + cones
for label, (path, color, off) in PATHS.items():
    px, py, pz = pcoords(path, elevation, off)
    sz = [elevation[p[0], p[1]] * 800 + 2 for p in path]

    # shadow
    fig1.add_trace(go.Scatter3d(
        x=px, y=py, z=sz, mode="lines",
        line=dict(color="rgba(0,0,0,0.25)", width=2),
        showlegend=False, hoverinfo="skip"
    ))
    # path
    fig1.add_trace(go.Scatter3d(
        x=px, y=py, z=pz, mode="lines",
        line=dict(color=color, width=5), name=label,
        hovertemplate="Step %{pointNumber}<br>Alt=%{z:.0f}m<extra>"+label+"</extra>"
    ))
    # direction cones
    step = max(1, len(path)//14)
    cx,cy,cz,cu,cv,cw = [],[],[],[],[],[]
    for i in range(0, len(path)-1, step):
        cx.append(path[i][1]);  cy.append(path[i][0])
        cz.append(elevation[path[i][0], path[i][1]] * 800 + off)
        cu.append(path[i+1][1]-path[i][1])
        cv.append(path[i+1][0]-path[i][0])
        cw.append(0)
    fig1.add_trace(go.Cone(
        x=cx, y=cy, z=cz, u=cu, v=cv, w=cw,
        colorscale=[[0,color],[1,color]],
        showscale=False, sizemode="absolute", sizeref=1.8,
        showlegend=False, hoverinfo="skip"
    ))

# Start / Goal
for label, pos, col, sym in [
    ("🟢 START", START, "#00ff44", "circle"),
    ("🔴 GOAL",  GOAL,  "#ff4444", "square")
]:
    fig1.add_trace(go.Scatter3d(
        x=[pos[1]], y=[pos[0]],
        z=[elevation[pos[0], pos[1]] * 800 + 90],
        mode="markers+text",
        marker=dict(size=12, color=col, symbol=sym,
                    line=dict(color="white", width=2)),
        text=[label], textposition="top center",
        textfont=dict(color="white", size=13), name=label
    ))

fig1.update_layout(
    title=dict(text="🚁 CEAADN — 3D Drone Flight over Mountain Terrain",
               font=dict(color="#00ffff", size=20), x=0.5),
    scene=dict(
        xaxis=dict(title="Col", showgrid=False, backgroundcolor="#0a0a1a",
                   tickfont=dict(color="#777")),
        yaxis=dict(title="Row", showgrid=False, backgroundcolor="#0a0a1a",
                   tickfont=dict(color="#777")),
        zaxis=dict(title="Altitude (m)", showgrid=True,
                   gridcolor="rgba(255,255,255,0.07)",
                   tickfont=dict(color="#777"), range=[0, 950]),
        bgcolor="#0a0a1a",
        camera=dict(eye=dict(x=1.7, y=-1.7, z=1.1),
                    center=dict(x=0, y=0, z=-0.05)),
        aspectratio=dict(x=1, y=1, z=0.45)
    ),
    paper_bgcolor="#0a0a1a",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(10,10,40,0.85)", bordercolor="#334", borderwidth=1),
    margin=dict(l=0, r=0, t=60, b=0),
    width=1050, height=700
)
fig1.show()

In [40]:
# ── VIZ 2 — 4-Panel 2D Heatmaps ─────────────────────────────────────────────
fig2 = make_subplots(
    rows=2, cols=2,
    subplot_titles=["Elevation", "Radar Detection", "GNN Risk", "Cost Map + Paths"],
    horizontal_spacing=0.08, vertical_spacing=0.12
)

maps   = [elevation, detection_map, gnn_risk, cost_map]
scales = ["Viridis", "Reds", "Plasma", "RdYlGn_r"]
for idx, (arr, sc) in enumerate(zip(maps, scales)):
    r, c = divmod(idx, 2)
    fig2.add_trace(go.Heatmap(z=arr, colorscale=sc, showscale=True), row=r+1, col=c+1)

# Paths on cost map panel (row=2, col=2)
for label, (path, color, _) in PATHS.items():
    fig2.add_trace(go.Scatter(
        x=[p[1] for p in path], y=[p[0] for p in path],
        mode="lines", line=dict(color=color, width=2), name=label
    ), row=2, col=2)

# Radar markers on all panels
for r in range(1, 3):
    for c in range(1, 3):
        for ri, rj in radar_positions:
            fig2.add_trace(go.Scatter(
                x=[rj], y=[ri], mode="markers",
                marker=dict(size=8, color="red", symbol="x"),
                showlegend=False, hoverinfo="skip"
            ), row=r, col=c)

fig2.update_layout(
    title=dict(text="📊 Environment Maps Overview",
               font=dict(color="#00ffff", size=17), x=0.5),
    paper_bgcolor="#0a0a1a", plot_bgcolor="#0a0a1a",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(10,10,40,0.85)", bordercolor="#334", borderwidth=1),
    width=1000, height=780
)
fig2.update_xaxes(showgrid=False)
fig2.update_yaxes(showgrid=False, autorange="reversed")
fig2.show()

In [41]:
# ── VIZ 3 — 2D Top-Down Path Comparison ─────────────────────────────────────
fig3 = go.Figure()
fig3.add_trace(go.Heatmap(z=cost_map, colorscale="RdYlGn_r", showscale=True,
                           colorbar=dict(title="Cost", tickfont=dict(color="white"),
                                         titlefont=dict(color="white"))))

lstyles = {"A* Baseline": "dot", "GNN-Guided A*": "solid", "PPO RL Agent": "solid"}
for label, (path, color, _) in PATHS.items():
    fig3.add_trace(go.Scatter(
        x=[p[1] for p in path], y=[p[0] for p in path],
        mode="lines", name=label,
        line=dict(color=color, width=3, dash=lstyles[label]),
        hovertemplate=label+"<br>Col=%{x}  Row=%{y}<extra></extra>"
    ))

for ri, rj in radar_positions:
    fig3.add_trace(go.Scatter(
        x=[rj], y=[ri], mode="markers+text",
        marker=dict(size=11, color="red", symbol="x"),
        text=["📡"], textposition="top center",
        showlegend=False
    ))

for label, pos, col, sym in [
    ("START", START, "#00ff44", "circle"),
    ("GOAL",  GOAL,  "#ff4444", "square")
]:
    fig3.add_trace(go.Scatter(
        x=[pos[1]], y=[pos[0]], mode="markers+text",
        marker=dict(size=14, color=col, symbol=sym,
                    line=dict(color="white", width=2)),
        text=[label], textposition="top center",
        textfont=dict(color="white"), showlegend=False
    ))

fig3.update_layout(
    title=dict(text="🗺️ 2D Top-Down Path Comparison on Cost Map",
               font=dict(color="#00ffff", size=17), x=0.5),
    xaxis=dict(title="Column", showgrid=False, tickfont=dict(color="#888")),
    yaxis=dict(title="Row",    showgrid=False, tickfont=dict(color="#888"),
               autorange="reversed"),
    paper_bgcolor="#0a0a1a", plot_bgcolor="#0a0a1a",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(10,10,40,0.85)", bordercolor="#334", borderwidth=1),
    width=800, height=700
)
fig3.show()

In [42]:
# ── VIZ 4 — Altitude Profile ─────────────────────────────────────────────────
fig4 = go.Figure()
for label, (path, color, _) in PATHS.items():
    alts = [elevation[p[0], p[1]] * 800 for p in path]
    prog = np.linspace(0, 100, len(alts))
    r, g, b = int(color[1:3],16), int(color[3:5],16), int(color[5:7],16)
    fig4.add_trace(go.Scatter(
        x=prog, y=alts, mode="lines", name=label,
        line=dict(color=color, width=2.5),
        fill="tozeroy", fillcolor=f"rgba({r},{g},{b},0.08)",
        hovertemplate="Progress=%{x:.1f}%<br>Alt=%{y:.0f}m<extra>"+label+"</extra>"
    ))
fig4.update_layout(
    title=dict(text="📈 Altitude Profile Along Each Path",
               font=dict(color="#00ffff", size=17), x=0.5),
    xaxis=dict(title="Path Progress (%)", gridcolor="rgba(255,255,255,0.08)",
               tickfont=dict(color="#aaa")),
    yaxis=dict(title="Terrain Altitude (m)", gridcolor="rgba(255,255,255,0.08)",
               tickfont=dict(color="#aaa")),
    paper_bgcolor="#0a0a1a", plot_bgcolor="#0a0a1a",
    font=dict(color="white"),
    legend=dict(bgcolor="rgba(10,10,40,0.85)", bordercolor="#334", borderwidth=1),
    width=950, height=420
)
fig4.show()

In [43]:

# ── VIZ 5 — Metrics Bar Chart ────────────────────────────────────────────────
fig5 = make_subplots(rows=1, cols=3,
                     subplot_titles=["Total Energy", "Radar Exposure", "Path Steps"])
methods = ["A* Baseline", "GNN A*", "PPO RL"]
vals    = [(e_as,e_gn,e_rl), (d_as,d_gn,d_rl), (l_as,l_gn,l_rl)]
colors  = ["#ff6b6b", "#00ffff", "#ffdd00"]

for ci, (v_tuple, title) in enumerate(zip(vals, ["Energy","Exposure","Steps"]), 1):
    fig5.add_trace(go.Bar(
        x=methods, y=list(v_tuple),
        marker_color=colors,
        text=[f"{v:.1f}" for v in v_tuple],
        textposition="outside",
        showlegend=False
    ), row=1, col=ci)

fig5.update_layout(
    title=dict(text="📊 Performance Comparison — A* vs GNN A* vs PPO RL",
               font=dict(color="#00ffff", size=17), x=0.5),
    paper_bgcolor="#0a0a1a", plot_bgcolor="#0a0a1a",
    font=dict(color="white", size=13),
    width=980, height=440
)
fig5.update_xaxes(tickfont=dict(color="white"))
fig5.update_yaxes(showgrid=True, gridcolor="rgba(255,255,255,0.08)")
fig5.show()

In [44]:
# ── VIZ 6 — GNN Loss Curve ───────────────────────────────────────────────────
fig6 = go.Figure()
fig6.add_trace(go.Scatter(
    y=losses, mode="lines",
    line=dict(color="#00d4ff", width=2),
    fill="tozeroy", fillcolor="rgba(0,212,255,0.10)",
    name="MSE Loss"
))
fig6.update_layout(
    title=dict(text="🧠 GNN Training Loss", font=dict(color="#00ffff", size=17), x=0.5),
    xaxis=dict(title="Epoch", gridcolor="rgba(255,255,255,0.08)", tickfont=dict(color="#aaa")),
    yaxis=dict(title="MSE",   gridcolor="rgba(255,255,255,0.08)", tickfont=dict(color="#aaa")),
    paper_bgcolor="#0a0a1a", plot_bgcolor="#0a0a1a",
    font=dict(color="white"),
    width=800, height=380
)
fig6.show()